In [ ]:
import cupy as cp
import torch
import triton
import triton.language as tl

In [ ]:
def layernorm_naive(x, weight, bias=None, eps=1e-6):

    *dims, embed_dim = x.shape

    if len(dims) > 1:
        x_reshaped = x.reshape(-1, embed_dim)
    else:
        x_reshaped = x

    mean = x_reshaped.mean(axis=-1, keepdims=True)
    var = x_reshaped.var(axis=-1, keepdims=True)
    normed = (x_reshaped - mean) / (cp.sqrt(var + eps))

    out = normed * weight.reshape(1, -1)
    if bias is not None:
        out = out + bias.reshape(1, -1)

    if len(dims) > 1:
        out = out.reshape(*dims, embed_dim)

    return out

In [ ]:
@triton.jit
def layernorm_kernel_forward_training(
        output_ptr,
        inv_var_ptr,                    # need for backward pass (N, ) # can be None during inference
        mean_ptr,                       # need for backward pass (N, ) # can be None during inference
        input_ptr,
        gamma_ptr,                      # 1D vector shared across all samples (E, ) can be None if no weight
        beta_ptr,                       # 1D vector shared across all samples (E, ) can be None if no bias
        input_row_stride,
        output_row_stride,
        DTYPE_FLAG: tl.constexpr,       # flag for if our data is float32 or float16
        eps: tl.constexpr,
        n_cols: tl.constexpr,           # dimensionality of our embeddings
        BLOCK_SIZE: tl.constexpr,       # closest power of 2 to our dim of embeddings
):
    row_idx = tl.program_id(0)

    # map ptrs to correct dtype
    pointer_dtype = tl.float32 if DTYPE_FLAG == 0 else tl.float16
    output_ptr = tl.cast(output_ptr, tl.pointer_type(pointer_dtype))
    input_ptr = tl.cast(input_ptr, tl.pointer_type(pointer_dtype))
    if gamma_ptr is not None:
        gamma_ptr = tl.cast(gamma_ptr, tl.pointer_type(pointer_dtype))
    if mean_ptr is not None:
        mean_ptr = tl.cast(mean_ptr, tl.pointer_type(pointer_dtype))
    if inv_var_ptr is not None:
        inv_var_ptr = tl.cast(inv_var_ptr, tl.pointer_type(pointer_dtype))
    if beta_ptr is not None:
        beta_ptr = tl.cast(beta_ptr, tl.pointer_type(pointer_dtype))

    row_start_ptr = input_ptr + row_idx * input_row_stride

    col_offsets = tl.arange(0, BLOCK_SIZE)
    mask = col_offsets < n_cols
    input_ptrs = row_start_ptr + col_offsets

    if gamma_ptr is not None:
        gamma_ptrs = gamma_ptr + col_offsets
        gammas = tl.load(gamma_ptrs, mask=mask, other=0.0)

    if beta_ptr is not None:
        beta_ptrs = beta_ptr + col_offsets
        betas = tl.load(beta_ptrs, mask=mask, other=0.0)

    row = tl.load(input_ptrs, mask=mask, other=0.0)

    row_mean = tl.sum(row, axis=0) / n_cols
    row_mean_centered = tl.where(mask, row - row_mean, 0.0)

    row_var = tl.sum(row_mean_centered * row_mean_centered, axis=0) / n_cols
    inv_var = tl.sqrt(row_var + eps)
    normed = row_mean_centered * inv_var

    if gamma_ptr is not None:
        output = normed * gammas

    else:
        output = normed

    if beta_ptr is not None:
        output += betas

    output_row_start_ptr = output_ptr + row_idx * output_row_stride
    output_ptrs = output_row_start_ptr + col_offsets
    tl.store(output_ptrs, output, mask=mask)


    if mean_ptr is not None and inv_var_ptr is not None:

        mean_row_ptrs = mean_ptr + row_idx
        tl.store(mean_row_ptrs, row_mean)

        inv_var_ptrs = inv_var_ptr + row_idx
        tl.store(inv_var_ptrs, inv_var)








In [ ]:
def fused_layernorm_forward(x, gamma=None, beta=None, eps=1e-5, training=True, use_dlpack=True):

    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)

    if use_dlpack:

        x = torch.utils.dlpack.from_dlpack(x)
        if gamma is not None:
            gamma = torch.utils.dlpcak.from_dlpack(gamma)
        if beta is not None:
            beta = torch.utils.dlpack.from_dlpack(beta)

        y = torch.empty_like(x)
        inv_var = torch.empty(n_rows, dtype=x.dtype, device=x.device) if training else None
        mean = torch.empty(n_rows, dtype=x.dtype, device=x.device) if training else None

        x_row_stride = x.stride(0)
        y_row_stride = y.stride(0)

        dtype_flag = 0 if x.dtype == torch.float32 else 1

        grid = (n_rows, )

        layernorm_kernel_forward_training[grid] (
            y,              # output_ptr
            inv_var,        # inv_var_ptr
            mean,           # mean_ptr
            x,              # input_ptr
            gamma,          # gamma_ptr
            beta,           # beta_ptr
            x_row_stride,   # input_row_stride
            y_row_stride,   # output_row_stride
            dtype_flag,     # dtype_flag (constexpr)
            eps,            # eps (constexpr)
            n_cols,         # n_cols (constexpr)
            BLOCK_SIZE,     # BLOCK_SIZE (constexpr)
        )

        # convert back to cupy if needed
        y = cp.from_dlpack(y)
        if training:
            mean = cp.from_dlpack(mean)
            inv_var = cp.from_dlpack(inv_var)
            return y, mean, inv_var
        return y


In [ ]:
@triton.jit
def layernorm_kernel_backward(
    x_ptr,
    gamma_ptr,
    mean_ptr,
    inv_var_ptr,
    dX_ptr,
    dGamma_ptr,
    dB_ptr,
    dY_ptr,
    stride_x,
    stride_dx,
    stride_dy,
    n_cols,
    BLOCK_SIZE: tl.constexpr,
    DTYPE_FLAG: tl.constexpr,
):

    row_idx = tl.program_id(0)

    # map pointers to correct Dtype
    pointer_type = tl.float32 if DTYPE_FLAG == 0 else tl.float16
    x_ptr = tl.cast(x_ptr, tl.pointer_type(pointer_type))
    mean_ptr = tl.cast(mean_ptr, tl.pointer_type(pointer_type))
    inv_var_ptr = tl.cast(inv_var_ptr, tl.pointer_type(pointer_type))
    dX_ptr = tl.cast(dX_ptr, tl.pointer_type(pointer_type))
    dY_ptr = tl.cast(dY_ptr, tl.pointer_type(pointer_type))

    if gamma_ptr is not None:
        gamma_ptr = tl.cast(gamma_ptr, tl.pointer_type(pointer_type))

    if dGamma_ptr is not None:
        dGamma_ptr = tl.cast(dGamma_ptr, tl.pointer_type(tl.float32))

    if dB_ptr is not None:
        dB_ptr = tl.cast(dB_ptr, tl.pointer_type(tl.float32))


    # indexes for this row we are processing
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols


    # load weights
    if gamma_ptr is not None:
        gamma = tl.load(gamma_ptr + cols, mask=mask, other=0.0).to(tl.float32)

    else:
        gamma = 1.0

    # get all pointers for this row
    row_x_ptr = x_ptr + row_idx * stride_x
    row_dX_ptr = dX_ptr + row_idx * stride_dx
    row_dY_ptr = dY_ptr + row_idx * stride_dy

    # 1d vectors
    row_mean_ptr = mean_ptr + row_idx
    row_inv_var_ptr = inv_var_ptr + row_idx

    x_row = tl.load(row_x_ptr + cols, mask=mask, other=0.0).to(tl.float32)
    dy_row = tl.load(row_dY_ptr + cols, mask=mask, other=0.0).to(tl.float32)
    row_mean = tl.load(row_mean_ptr).to(tl.float32)
    row_inv_std = tl.load(row_inv_var_ptr).to(tl.float32)

    x_hat = (x_row - row_mean) * row_inv_std
    wdy = gamma * dy_row        # <- upstream grad (dL/dy)(dy/dx_hat)
    mean1 = tl.sum(x_hat * wdy, axis=0) / n_cols
    mean2 = tl.sum(wdy, axis=0) / n_cols
    dx = row_inv_std * (wdy - (x_hat * mean1 + mean2))

    tl.store(row_dX_ptr + cols, dx.to(pointer_type), mask=mask)

    if dGamma_ptr is not None:
        dW = dy_row * x_hat
        tl.atomic_hat(dGamma_ptr + cols, dW, mask=mask)

    if dB_ptr is not None:
        db = dy_row
        tl.atomic_add(dB_ptr + cols, db, mask=mask)

In [ ]:
def fused_layernorm_backward(x, mean, inv_var, dy, gamma-None, beta=None, use_dlpack=True):
    n_rows, n_cols = dy.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)

    if use_dlpack:
        x = torch.utils.dlpack.from_dlpack(x)
        dy = torch.utils.dlpack.from_dlpack(dy)
        mean = torch.utils.dlpack.from_dlpack(mean)
        inv_var = torch.utils.dlpack.from_dlpack(inv_var)
        gamma = torch.utils.dlpack.from_dlpack(gamma) if gamma is not None else None
        beta = torch.utils.dlpack.from_dlpack(beta) if beta is not None else None

        grad_dtype = torch.float32
        dx = torch.empty_like(dy)
        dgamma = torch.zeros(n_cols, dtype=grad_dtype, device=dy.device) if gamma is not None else None
        dbeta = torch.zeros(n_cols, dtype=grad_dtype, device=dy.device) if beta is not None else None

        stride_x = x.stride(0)
        stride_dx = dx.stride(0)
        stride_dy = dy.stride(0)
        dtype_flag = 0 if dy.dtype == torch.float32 else 1

        grid = (n_rows, )
        layernorm_kernel_backward[grid](
            x, gamma, mean, inv_var, dx, dgamma, dbeta, dy,
            stride_x, stride_dx, stride_dy, n_cols, BLOCK_SIZE, dtype_flag,
        )

        dx = cp.from_dlpack(dx)

        # convert back to original dtype
        if gamma is not None:
            dgamma = cp.from_dlpack(dgamma.to(gamma.dtype))

        if beta is not None:
            dbeta = cp.from_dlpack(dbeta.do(beta.dtype))


        return dx, dgamma, dbeta